# T16 CUDA throughput smoke-03

Owner-run throughput evidence only. This notebook does not run a quality gate or serving qualification. Choose a Colab GPU runtime, then use **Runtime → Run all**. P0 runs the isolated learner benchmark first, then the pre-registered smoke-03: seven registered 300 s rows, with six scored by default under the 3,600 s runner cap (only row 5 is omitted), and one separate unscored 60 s cProfile capture on row 4. Completed P0 receipts and resumable smoke artifacts are written to Drive; failed setup is reported separately.

In [ ]:
# Parameters. Change RUN_NAME only to start a separate, non-resumed smoke.
PINNED_COMMIT = 'ef32fb99e7691fa3e716083a0abb94adf3eea2a0'
RUN_NAME = 'smoke-03'
MATRIX = 'smoke03'
DRIVE_ROOT = '/content/drive/MyDrive/t16_colab'
INPUT_ARCHIVE = f'{DRIVE_ROOT}/t16_colab_inputs.tar.gz'
EXPECTED_ARCHIVE_SHA256 = '229be2949c99f198b27decdf403b83ec4085d8df2dcfe629b7eff9208593be8e'
RESULTS_DIR = f'{DRIVE_ROOT}/{RUN_NAME}'
REPOSITORY = 'https://github.com/josenunezwork/snake-dqn.git'
CUDA_WHEEL_INDEX = 'https://download.pytorch.org/whl/cu126'
TOTAL_BUDGET_SECONDS = 4500  # setup, P0, parity, warm-up, profile, and seven 300 s rows
TIMING_SECONDS = 300
SCRATCH_DIR = '/content/t16-colab-scratch'  # local disk for trainer outputs; copied to Drive per run


In [ ]:
import hashlib
import json
import os
import signal
import shutil
import subprocess
import sys
import time
import tempfile
from pathlib import Path

from google.colab import drive, files

drive.mount('/content/drive')
started_after_drive_auth = time.monotonic()
drive_root = Path(DRIVE_ROOT)
results_dir = Path(RESULTS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)
result_path = results_dir / 'colab_throughput_results.json'
invocation_path = results_dir / 'invocation_result.json'
report_path = result_path
deadline = started_after_drive_auth + TOTAL_BUDGET_SECONDS

def cleanup_owned_group(process, grace_seconds):
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    stop_at = min(deadline, time.monotonic() + grace_seconds)
    while time.monotonic() < stop_at:
        process.poll()
        try:
            os.killpg(process.pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.1)
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass

def run_owned(command, *, check=True, capture_output=False, cleanup_grace=5, deadline_at=None, **kwargs):
    active_deadline = min(deadline, deadline_at or deadline)
    timeout = active_deadline - time.monotonic() - cleanup_grace
    if timeout <= 0:
        raise TimeoutError(f'The {TOTAL_BUDGET_SECONDS}-second budget expired before this subprocess could start.')
    print('+', ' '.join(map(str, command)), flush=True)
    if capture_output:
        kwargs['stdout'] = subprocess.PIPE
    process = subprocess.Popen(command, text=True, start_new_session=True, **kwargs)
    try:
        stdout, _ = process.communicate(timeout=timeout)
    except BaseException as error:
        cleanup_owned_group(process, cleanup_grace)
        try:
            process.wait(timeout=max(0.1, min(1, deadline - time.monotonic())))
        except subprocess.TimeoutExpired:
            raise RuntimeError(f'Owned subprocess did not exit after cleanup: {command[0]}') from error
        if isinstance(error, subprocess.TimeoutExpired):
            raise TimeoutError(f'Subprocess exceeded the total {TOTAL_BUDGET_SECONDS}-second budget: {command[0]}') from error
        raise
    cleanup_owned_group(process, cleanup_grace)
    completed = subprocess.CompletedProcess(command, process.returncode, stdout=stdout)
    if check and completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, command, output=stdout)
    return completed

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

interpreter_info = {}
try:
    archive = Path(INPUT_ARCHIVE)
    if not archive.is_file():
        raise FileNotFoundError(f'Missing Drive input archive: {archive}')
    if sha256(archive) != EXPECTED_ARCHIVE_SHA256:
        raise RuntimeError('Input archive SHA-256 differs from the published bundle; re-upload it unchanged.')

    checkout = Path('/content/t16-snake-dqn')
    venv = Path('/content/t16-colab-venv')
    shutil.rmtree(checkout, ignore_errors=True)
    shutil.rmtree(venv, ignore_errors=True)
    run_owned(['git', 'init', str(checkout)])
    run_owned(['git', '-C', str(checkout), 'remote', 'add', 'origin', REPOSITORY])
    run_owned(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', PINNED_COMMIT])
    run_owned(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'])
    actual_commit = run_owned(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], capture_output=True).stdout.strip()
    if actual_commit != PINNED_COMMIT:
        raise RuntimeError(f'Pinned checkout mismatch: expected {PINNED_COMMIT}, got {actual_commit}')

    # The 3.13 notebook kernel only orchestrates; uv supplies the workload interpreter.
    sys.path.insert(0, str(checkout))
    from research.t16_colab.environment import create_environment
    python, interpreter_info = create_environment(venv, run_owned)
    # requirements-gpu carries the repo pins; keep NumPy/Numba ABI pins explicit.
    run_owned([python, '-m', 'pip', 'install', '-r', str(checkout / 'requirements-gpu.txt'),
                 'numpy==1.26.4', 'numba==0.68.0', 'llvmlite==0.50.0', 'pytest>=9,<10'])
    # A fresh 3.12 venv cannot reuse the kernel's 3.13 torch wheel.
    run_owned([python, '-m', 'pip', 'install', '--index-url', CUDA_WHEEL_INDEX,
                 'torch==2.9.1'])
    run_owned([python, '-c',
        "import torch; assert torch.__version__.split('+')[0].startswith('2.9.') and torch.cuda.is_available(), torch.__version__"])

    extracted_parent = Path('/content/t16-colab-extracted')
    shutil.rmtree(extracted_parent, ignore_errors=True)
    verifier = (
        'from pathlib import Path; from research.t16_colab.bundle import verify_and_extract; '
        'print(verify_and_extract(Path(' + repr(str(archive)) + '), Path(' + repr(str(extracted_parent)) + ')))'
    )
    run_owned([python, '-c', verifier], cwd=checkout)
    input_root = extracted_parent / 't16-colab-inputs'

    # P0 is deliberately before smoke writes: isolate host gather and GPU update legs.
    # Each benchmark writes its own result. The P0 receipt records failures too, so a
    # diagnostic failure cannot suppress the independently useful smoke matrix.
    p0_dir = results_dir / 'p0_learner_bench' / PINNED_COMMIT
    runtime_probe = "import json, torch; print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'capability': torch.cuda.get_device_capability(0)}))"
    p0_runtime = json.loads(run_owned([python, '-c', runtime_probe], capture_output=True).stdout)
    p0_identity = {'commit': actual_commit, 'archive_sha256': EXPECTED_ARCHIVE_SHA256, 'runtime': p0_runtime}
    p0_identity_path = p0_dir / 'identity.json'
    if p0_identity_path.exists() and json.loads(p0_identity_path.read_text()) != p0_identity:
        raise RuntimeError('P0 output belongs to different code or inputs; choose a new RUN_NAME.')
    p0_dir.mkdir(parents=True, exist_ok=True)
    p0_identity_path.write_text(json.dumps(p0_identity, indent=2) + '\n')
    p0_receipt_path = p0_dir / 'p0_receipt.json'
    p0_receipt = {
        'schema': 't17b-p0-learner-bench-receipt/v1',
        'identity': p0_identity,
        'status': 'running',
        'benchmarks': [],
    }
    def write_p0_receipt():
        p0_receipt['updated_after_drive_auth_seconds'] = time.monotonic() - started_after_drive_auth
        p0_receipt_path.write_text(json.dumps(p0_receipt, indent=2, sort_keys=True) + '\n')
    write_p0_receipt()
    p0_deadline = time.monotonic() + 300
    shm_df = run_owned(['df', '-h', '/dev/shm'], capture_output=True).stdout
    (p0_dir / 'df_dev_shm.txt').write_text(shm_df)
    p0_configs = 't1_k2_c_ro,t1_bf16_k2_c_ro,t1_k4_c_ro,t1_bf16_k4_c_ro'
    local_ring_dir = Path(SCRATCH_DIR) / 'p0-rings'
    local_ring_dir.mkdir(parents=True, exist_ok=True)
    for ring_dir in (Path('/dev/shm'), local_ring_dir):
        for pack_flag, pack_name in (('--obs-pack', 'packed'), ('--raw-rings', 'raw')):
            for gather_threads in (0, 3):
                stem = f'{pack_name}_{ring_dir.name}_g{gather_threads}'
                with tempfile.TemporaryDirectory(prefix="t17b-p0-", dir=ring_dir) as owned_ring_dir:
                    p0 = [python, 'research/perf_learner_20261013/bench_learner.py',
                          '--devices', 'cuda', '--configs', p0_configs, pack_flag,
                          '--ring-dir', owned_ring_dir, '--gather-threads', str(gather_threads),
                          '--updates', '30', '--rounds', '5', '--warm', '8',
                          '--out', str(p0_dir / f'{stem}.json'),
                          '--note', 't17b P0; report host_gather_ms and gpu_leg_ms separately']
                    benchmark = {
                        'name': stem, 'obs_layout': pack_name, 'ring_dir': str(ring_dir),
                        'gather_threads': gather_threads, 'output': str(p0_dir / f'{stem}.json'),
                    }
                    started_benchmark = time.monotonic()
                    try:
                        run_owned(p0, cwd=checkout, deadline_at=p0_deadline)
                        benchmark['status'] = 'completed'
                    except Exception as error:
                        benchmark['status'] = 'failed'
                        benchmark['error'] = f'{type(error).__name__}: {error}'
                        print(f'P0 benchmark {stem} failed; continuing to smoke matrix: {benchmark["error"]}', file=sys.stderr)
                    finally:
                        benchmark['wall_seconds'] = time.monotonic() - started_benchmark
                        p0_receipt['benchmarks'].append(benchmark)
                        write_p0_receipt()
    p0_receipt['status'] = (
        'completed_with_failures'
        if any(benchmark['status'] == 'failed' for benchmark in p0_receipt['benchmarks'])
        else 'completed'
    )
    write_p0_receipt()

    remaining_budget = min(3600, int(deadline - time.monotonic() - 12))
    if remaining_budget <= 0:
        raise TimeoutError(f'The {TOTAL_BUDGET_SECONDS}-second budget expired during setup.')
    # A clean exit can take the runner grace. Seven rows need 3785 s at that conservative
    # allowance, while six (omitting only row 5) need 3335 s and fit the 3600 s cap.
    # Once selected, retain the row set on reconnect so its identity cannot change.
    full_matrix_estimated_seconds = 3785
    six_row_estimated_seconds = 3335
    runner_identity_path = results_dir / 'identity.json'
    if runner_identity_path.exists():
        prior_identity = json.loads(runner_identity_path.read_text())
        smoke_args = ['--omit-smoke03-row5'] if prior_identity.get('omitted_smoke03_row5') else []
    else:
        smoke_args = ['--omit-smoke03-row5'] if remaining_budget < full_matrix_estimated_seconds else []
    print('smoke-03 plan:', 'six rows (row 5 omitted)' if smoke_args else 'seven rows',
          'estimated runner seconds:', six_row_estimated_seconds if smoke_args else full_matrix_estimated_seconds,
          'available:', remaining_budget)
    run = [python, 'research/t16_colab/run_smoke.py', '--inputs', str(input_root),
           '--output', str(results_dir), '--commit', PINNED_COMMIT, '--matrix', MATRIX, '--seconds', str(TIMING_SECONDS),
           '--budget-seconds', str(remaining_budget), '--scratch-dir', SCRATCH_DIR, *smoke_args]
    invocation_path.unlink(missing_ok=True)
    completed = run_owned(run, cwd=checkout, check=False, cleanup_grace=12)
    if completed.returncode and not invocation_path.exists():
        raise RuntimeError(f'runner exited {completed.returncode} before writing its invocation result')
    report_path = Path('/content/t16-colab-current/colab_throughput_results.json')
    report_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(invocation_path, report_path)
except Exception as error:
    failure = {
        'schema': 't16-colab-throughput/v1', 'commit': PINNED_COMMIT, 'configs': [],
        **interpreter_info, 'kernel_python': sys.version,
        'status': 'failed', 'error': f'{type(error).__name__}: {error}',
        'authority': 'throughput smoke only; not gate, quality, or serving evidence',
        'invocation_wall_seconds': time.monotonic() - started_after_drive_auth,
    }
    result_path = Path('/content/t16-colab-failure/colab_throughput_results.json')
    result_path.parent.mkdir(parents=True, exist_ok=True)
    result_path.write_text(json.dumps(failure, indent=2) + '\n')
    report_path = result_path
    print(failure['error'], file=sys.stderr)
finally:
    print(f'Results: {report_path}')
    if report_path.exists():
        display(json.loads(report_path.read_text()))
        files.download(str(report_path))
